In [1]:
import requests
import pandas as pd


GBFS_URL = "https://gbfs.citibikenyc.com/gbfs/2.3/gbfs.json"


def get_gbfs_feeds() -> dict[str, str]:
    """Загружает список всех доступных GBFS фидов и возвращает {name: url}."""
    resp = requests.get(GBFS_URL, timeout=10)
    resp.raise_for_status()
    feeds = resp.json()["data"]["en"]["feeds"]
    return {f["name"]: f["url"] for f in feeds}


def fetch_feed(url: str) -> dict:
    """Загружает данные конкретного фида."""
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()["data"]


def load_station_information(feeds: dict[str, str]) -> pd.DataFrame:
    """Станции с координатами, вместимостью и т.д."""
    data = fetch_feed(feeds["station_information"])
    return pd.DataFrame(data["stations"])


def load_station_status(feeds: dict[str, str]) -> pd.DataFrame:
    """Текущий статус станций: доступные велосипеды, доки и т.д."""
    data = fetch_feed(feeds["station_status"])
    return pd.DataFrame(data["stations"])


def load_realtime_snapshot(feeds: dict[str, str]) -> pd.DataFrame:
    """Объединяет station_information + station_status в один DataFrame."""
    info = load_station_information(feeds)
    status = load_station_status(feeds)
    return info.merge(status, on="station_id", suffixes=("", "_status"))


if __name__ == "__main__":
    feeds = get_gbfs_feeds()
    print("Доступные фиды:", list(feeds.keys()))

    df = load_realtime_snapshot(feeds)
    print(f"\nСтанций: {len(df)}")
    print(f"Колонки: {list(df.columns)}")
    print(f"\nВсего доступных велосипедов: {df['num_bikes_available'].sum()}")
    print(f"Всего свободных доков: {df['num_docks_available'].sum()}")
    print(df[["name", "lat", "lon", "capacity", "num_bikes_available", "num_docks_available"]].head(10))

Доступные фиды: ['gbfs', 'system_information', 'station_information', 'station_status', 'free_bike_status', 'system_hours', 'system_calendar', 'system_regions', 'system_pricing_plans', 'system_alerts', 'gbfs_versions', 'vehicle_types']

Станций: 2356
Колонки: ['region_id', 'lat', 'lon', 'station_id', 'rental_uris', 'capacity', 'short_name', 'name', 'num_docks_available', 'is_returning', 'last_reported', 'vehicle_types_available', 'num_bikes_available', 'is_installed', 'is_renting', 'num_ebikes_available', 'num_docks_disabled', 'num_bikes_disabled', 'num_scooters_available', 'num_scooters_unavailable']

Всего доступных велосипедов: 26895
Всего свободных доков: 36848
                              name        lat        lon  capacity  \
0                  W 25 St & 6 Ave  40.743954 -73.991449        51   
1             43 St & Skillman Ave  40.746927 -73.920825        19   
2                 31 St & Broadway  40.762110 -73.925230        35   
3        E Gun Hill Rd & Tryon Ave  40.880200 

In [2]:
df

,region_id,lat,lon,station_id,rental_uris,capacity,short_name,name,num_docks_available,is_returning,last_reported,vehicle_types_available,num_bikes_available,is_installed,is_renting,num_ebikes_available,num_docks_disabled,num_bikes_disabled,num_scooters_available,num_scooters_unavailable
0,71,40.743954,-73.991449,66dc2995-0aca-11e7-82f6-3863bb44ef7c,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,51,6215.04,W 25 St & 6 Ave,0,0,86400,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,0,0,0,0,0,NaN,NaN
1,71,40.746927,-73.920825,41495491-5d89-4e14-aab9-c3db04aad399,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,19,6325.01,43 St & Skillman Ave,0,0,86400,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,0,0,0,0,0,NaN,NaN
2,71,40.762110,-73.925230,1905837242740508940,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,35,6789.20,31 St & Broadway,0,0,86400,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,0,0,0,0,0,NaN,NaN
3,71,40.880200,-73.876970,2171904223419888148,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,0,8821.07,E Gun Hill Rd & Tryon Ave,0,0,86400,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,0,0,0,0,0,NaN,NaN
4,71,40.883360,-73.915050,2177014969129222184,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,0,8838.06,Henry Hudson Pkwy E & W 231 St,0,0,86400,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,0,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2351,70,40.692770,-74.087800,2116636221383887448,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,18,JC139,Danforth Light Rail,5,1,1772998234,"[{'count': 13, 'vehicle_type_id': '1'}, {'coun...",13,1,1,0,0,0,0.0,0.0
2352,311,40.745984,-74.028199,519824e4-69ba-4270-a395-17c204f328f8,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,21,HB603,8 St & Washington St,5,0,1772961455,"[{'count': 0, 'vehicle_type_id': '1'}, {'count...",0,1,0,0,0,15,0.0,0.0
2353,311,40.750604,-74.024020,bd6f422b-d7ae-4d7e-9261-653fdd8e6888,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,28,HB201,12 St & Sinatra Dr N,14,1,1772998264,"[{'count': 14, 'vehicle_type_id': '1'}, {'coun...",14,1,1,0,0,0,0.0,0.0
2354,70,40.704503,-74.077210,2108373961268485100,{'android': 'https://bkn.lft.to/lastmile_qr_sc...,14,JC127,Arlington Ave & Wilkinson Ave,9,1,1772998273,"[{'count': 5, 'vehicle_type_id': '1'}, {'count...",5,1,1,0,0,0,0.0,0.0
